In [1]:
"""
============================================================
  EnViT5 + LoRA → FastAPI Inference Server trên Kaggle
============================================================
Hướng dẫn sử dụng:
  1. Upload thư mục LoRA weights lên Kaggle Dataset hoặc dùng đường dẫn
     mặc định `/kaggle/input/<dataset-name>/lora_weights/`.
  2. Thay LORA_WEIGHTS_PATH bằng đường dẫn thực tế.
  3. Đặt NGROK_AUTH_TOKEN bằng token của bạn từ https://dashboard.ngrok.com
  4. Chạy toàn bộ cell trong Kaggle Notebook.
============================================================
"""

'\n============================================================\n  EnViT5 + LoRA → FastAPI Inference Server trên Kaggle\n============================================================\nHướng dẫn sử dụng:\n  1. Upload thư mục LoRA weights lên Kaggle Dataset hoặc dùng đường dẫn\n     mặc định `/kaggle/input/<dataset-name>/lora_weights/`.\n  2. Thay LORA_WEIGHTS_PATH bằng đường dẫn thực tế.\n  3. Đặt NGROK_AUTH_TOKEN bằng token của bạn từ https://dashboard.ngrok.com\n  4. Chạy toàn bộ cell trong Kaggle Notebook.\n============================================================\n'

In [2]:
# CELL 1 — Cài đặt thư viện
!pip install -U peft datasets sacrebleu sentencepiece accelerate evaluate bitsandbytes transformers==4.44.0 pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 75.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.9 MB/s eta 0:00:00:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface

In [3]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [4]:
!find /kaggle/input -name "added_tokens.json"

/kaggle/input/datasets/ctrungnguyn123/envit5-lora-model/added_tokens.json


In [ ]:

# CELL 2 — Cấu hình
import os

# ── Đường dẫn ─────────────────────────────────
BASE_MODEL_NAME   = "VietAI/envit5-translation"          # HuggingFace model ID
LORA_WEIGHTS_PATH = "/kaggle/input/datasets/ctrungnguyn123/envit5-lora-model"  

# ── Ngrok ──────────────────────────────────────
NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN", "your-token-here")  # Thay bằng token của bạn nếu không dùng biến môi trường

# ── Server ─────────────────────────────────────
PORT = 8000

# ── Inference ──────────────────────────────────
MAX_NEW_TOKENS   = 512
NUM_BEAMS        = 4
LENGTH_PENALTY   = 1.0
EARLY_STOPPING   = True

print(" Cấu hình xong.")
print(f"   Base model : {BASE_MODEL_NAME}")
print(f"   LoRA path  : {LORA_WEIGHTS_PATH}")

 Cấu hình xong.
   Base model : VietAI/envit5-translation
   LoRA path  : /kaggle/input/datasets/ctrungnguyn123/envit5-lora-model


In [7]:

# CELL 3 — Load & Merge model
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ── Phát hiện thiết bị ───────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = DEVICE == "cuda"
print(f"  Thiết bị: {DEVICE.upper()}  |  FP16: {USE_FP16}")


# ── Base model ───────────────────────────────
print(" Đang tải base model...")
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16 if USE_FP16 else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
)

  Thiết bị: CUDA  |  FP16: True
 Đang tải base model...


config.json:   0%|          | 0.00/721 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1598: UserWarning: The following device_map keys do not match any submodules in the model: ['decoder.embed_tokens', 'encoder.embed_tokens']
  warnings.warn(


In [8]:
# ── Tokenizer ────────────────────────────────
print(" Đang tải tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME,
    use_fast=False,
)

 Đang tải tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [9]:
from peft import PeftModel
# ── Merge LoRA ───────────────────────────────
print(" Đang merge LoRA weights...")
if os.path.isdir(LORA_WEIGHTS_PATH):
    model = PeftModel.from_pretrained(base_model, LORA_WEIGHTS_PATH)
    model = model.merge_and_unload()          # merge → full model thuần
    print(" Merge LoRA thành công.")
else:
    print("  Không tìm thấy LoRA weights — dùng base model thuần.")
    model = base_model

# ── Chuyển sang eval mode ────────────────────
model.eval()
if DEVICE == "cpu":
    model = model.to(DEVICE)

print(f" Model sẵn sàng trên {DEVICE.upper()}.")

 Đang merge LoRA weights...
 Merge LoRA thành công.
 Model sẵn sàng trên CUDA.


In [10]:

# CELL 4 — Hàm inference
def run_inference(
    text: str,
    max_new_tokens: int = MAX_NEW_TOKENS,
    num_beams: int = NUM_BEAMS,
    length_penalty: float = LENGTH_PENALTY,
    early_stopping: bool = EARLY_STOPPING,
) -> str:
    """
    Chạy seq2seq inference và trả về text đã dịch / sinh ra.
    """
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            length_penalty=length_penalty,
            early_stopping=early_stopping,
        )

    decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
    return decoded[0] if decoded else ""


# Kiểm tra nhanh
_sample = "Bệnh nhân bị đau đầu và sốt cao."
print(" Kiểm tra inference:")
print(" Input :", _sample)
print(" Output:", run_inference(_sample))

 Kiểm tra inference:
 Input : Bệnh nhân bị đau đầu và sốt cao.


2026-05-18 14:46:20.789802: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779115580.998364      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779115581.058190      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779115581.535918      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779115581.535959      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779115581.535962      57 computation_placer.cc:177] computation placer alr

 Output: en: The patient suffered from headache and high fever.


In [11]:
# ─────────────────────────────────────────────
# CELL 5 — FastAPI app
# ─────────────────────────────────────────────
# %%
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import Optional
import time

app = FastAPI(
    title="EnViT5-LoRA Inference API",
    description="Medical En↔Vi translation powered by EnViT5 fine-tuned with LoRA on MedEV dataset.",
    version="1.0.0",
)

# ── CORS (cho phép gọi từ bất kỳ origin nào) ─
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Schema ────────────────────────────────────
class PredictRequest(BaseModel):
    text: str = Field(
        ...,
        min_length=1,
        max_length=2048,
        description="Văn bản đầu vào cần xử lý (dịch hoặc sinh).",
        examples=["The patient has a headache and high fever."],
    )
    max_new_tokens: Optional[int] = Field(
        default=MAX_NEW_TOKENS,
        ge=1, le=1024,
        description="Số token tối đa được sinh ra.",
    )
    num_beams: Optional[int] = Field(
        default=NUM_BEAMS,
        ge=1, le=10,
        description="Beam search width.",
    )
    length_penalty: Optional[float] = Field(
        default=LENGTH_PENALTY,
        ge=0.1, le=5.0,
        description="Hệ số phạt độ dài (>1 ưu tiên câu dài hơn).",
    )


class PredictResponse(BaseModel):
    output: str
    input_length: int
    output_length: int
    latency_ms: float


# ── Endpoints ────────────────────────────────

@app.get("/", tags=["Health"])
async def root():
    return {
        "status": "running",
        "model": BASE_MODEL_NAME,
        "device": DEVICE,
        "fp16": USE_FP16,
    }


@app.get("/health", tags=["Health"])
async def health():
    return {"status": "ok"}


@app.post("/predict", response_model=PredictResponse, tags=["Inference"])
async def predict(request: PredictRequest):
    """
    Nhận một đoạn văn bản và trả về kết quả từ model EnViT5-LoRA.

    - **text**: văn bản đầu vào (EN hoặc VI)
    - **max_new_tokens**: số token tối đa sinh ra (mặc định 512)
    - **num_beams**: beam search width (mặc định 4)
    - **length_penalty**: hệ số phạt độ dài (mặc định 1.0)
    """
    try:
        t0 = time.perf_counter()
        output_text = run_inference(
            text=request.text,
            max_new_tokens=request.max_new_tokens,
            num_beams=request.num_beams,
            length_penalty=request.length_penalty,
        )
        latency_ms = (time.perf_counter() - t0) * 1000

        return PredictResponse(
            output=output_text,
            input_length=len(request.text),
            output_length=len(output_text),
            latency_ms=round(latency_ms, 2),
        )
    except RuntimeError as e:
        # GPU OOM hoặc lỗi model
        raise HTTPException(status_code=500, detail=f"Model inference lỗi: {e}")
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))


@app.post("/predict/batch", tags=["Inference"])
async def predict_batch(requests: list[PredictRequest]):
    """
    Dự đoán cho nhiều input cùng lúc (tối đa 16).
    """
    if len(requests) > 16:
        raise HTTPException(status_code=400, detail="Tối đa 16 request mỗi batch.")

    results = []
    for req in requests:
        t0 = time.perf_counter()
        output_text = run_inference(
            text=req.text,
            max_new_tokens=req.max_new_tokens,
            num_beams=req.num_beams,
            length_penalty=req.length_penalty,
        )
        latency_ms = (time.perf_counter() - t0) * 1000
        results.append(PredictResponse(
            output=output_text,
            input_length=len(req.text),
            output_length=len(output_text),
            latency_ms=round(latency_ms, 2),
        ))
    return results


print(" FastAPI app đã khởi tạo.")
print(" Endpoints: GET /health  |  POST /predict  |  POST /predict/batch")

 FastAPI app đã khởi tạo.
 Endpoints: GET /health  |  POST /predict  |  POST /predict/batch


In [12]:
# ─────────────────────────────────────────────
# CELL 6 — Khởi động ngrok + uvicorn
# ─────────────────────────────────────────────
# %%
import nest_asyncio
import uvicorn
import threading
from pyngrok import ngrok, conf

nest_asyncio.apply()  # Cho phép asyncio chạy trong Jupyter/Kaggle

# ── Cấu hình ngrok ────────────────────────────
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Đóng tunnel cũ nếu có
ngrok.kill()

# Mở tunnel mới
tunnel = ngrok.connect(PORT, "http")
public_url = tunnel.public_url

print("=" * 55)
print(" PUBLIC URL (chia sẻ URL này cho máy local):")
print(f"   {public_url}")
print("=" * 55)
print(f" Thử ngay:")
print(f"   curl -X POST {public_url}/predict \\")
print(f'        -H "Content-Type: application/json" \\')
print(f'        -d \'{{"text": "The patient has a headache."}}\'\n')

# ── Khởi động uvicorn trong background thread ─
def run_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=PORT,
        log_level="info",
    )

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("Server đang chạy trên cổng", PORT)
print("Nhấn  Stop (interrupt kernel) để dừng server.\n")
print("Swagger UI :", public_url + "/docs")
print("ReDoc UI   :", public_url + "/redoc")

 PUBLIC URL (chia sẻ URL này cho máy local):
   https://subplot-strep-ragweed.ngrok-free.dev
 Thử ngay:
   curl -X POST https://subplot-strep-ragweed.ngrok-free.dev/predict \
        -H "Content-Type: application/json" \
        -d '{"text": "The patient has a headache."}'

Server đang chạy trên cổng 8000
Nhấn  Stop (interrupt kernel) để dừng server.

Swagger UI : https://subplot-strep-ragweed.ngrok-free.dev/docs
ReDoc UI   : https://subplot-strep-ragweed.ngrok-free.dev/redoc


In [13]:
# ─────────────────────────────────────────────
# CELL 7 — Test nhanh từ bên trong Notebook
# ─────────────────────────────────────────────
# %%
import requests as http_requests
import json
import time

time.sleep(2)  # Chờ server khởi động

# ── Test /health ──────────────────────────────
r = http_requests.get(f"http://localhost:{PORT}/health")
print("GET /health →", r.json())

# ── Test /predict (EN → VI) ───────────────────
payload_en = {
    "text": "The patient presents with acute myocardial infarction and requires immediate intervention.",
    "num_beams": 4,
}
r = http_requests.post(f"http://localhost:{PORT}/predict", json=payload_en)
result = r.json()
print("\nPOST /predict (EN→VI):")
print(f"  Input  : {payload_en['text']}")
print(f"  Output : {result.get('output', 'N/A')}")
print(f"  Latency: {result.get('latency_ms', 'N/A')} ms")

# ── Test /predict (VI → EN) ───────────────────
payload_vi = {
    "text": "Bệnh nhân bị tiểu đường type 2 cần điều chỉnh liều insulin.",
    "num_beams": 4,
}
r = http_requests.post(f"http://localhost:{PORT}/predict", json=payload_vi)
result = r.json()
print("\nPOST /predict (VI→EN):")
print(f"  Input  : {payload_vi['text']}")
print(f"  Output : {result.get('output', 'N/A')}")
print(f"  Latency: {result.get('latency_ms', 'N/A')} ms")

print("\n Tất cả test đều thành công!")
print(f"\n Public URL để dùng từ máy local:\n   {public_url}/predict")

INFO:     Started server process [57]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:49682 - "GET /health HTTP/1.1" 200 OK
GET /health → {'status': 'ok'}
INFO:     127.0.0.1:49690 - "POST /predict HTTP/1.1" 200 OK

POST /predict (EN→VI):
  Input  : The patient presents with acute myocardial infarction and requires immediate intervention.
  Output : vi: Bệnh nhân có biểu hiện nhồi máu cơ tim cấp và cần can thiệp ngay lập tức.
  Latency: 776.73 ms
INFO:     127.0.0.1:49698 - "POST /predict HTTP/1.1" 200 OK

POST /predict (VI→EN):
  Input  : Bệnh nhân bị tiểu đường type 2 cần điều chỉnh liều insulin.
  Output : en: Patients with type 2 diabetes need to adjust their insulin dosage.
  Latency: 592.29 ms

 Tất cả test đều thành công!

 Public URL để dùng từ máy local:
   https://subplot-strep-ragweed.ngrok-free.dev/predict


In [14]:
# ─────────────────────────────────────────────
# CELL 8 — Ví dụ gọi API từ máy local (Python)
# ─────────────────────────────────────────────
# %%
# ─── Chạy đoạn này trên MÁY LOCAL của bạn ───
EXAMPLE_LOCAL_USAGE = '''
# ============================================================
# Chạy đoạn này trên MÁY LOCAL (không phải Kaggle)
# ============================================================
import requests

PUBLIC_URL = "https://xxxx-xxxx.ngrok-free.app"  # ← Thay bằng URL thực

response = requests.post(
    f"{PUBLIC_URL}/predict",
    json={
        "text": "Patient has symptoms of pneumonia.",
        "max_new_tokens": 256,
        "num_beams": 4,
        "length_penalty": 1.0,
    },
    headers={"Content-Type": "application/json"},
    timeout=60,
)

data = response.json()
print("Kết quả:", data["output"])
print("Latency:", data["latency_ms"], "ms")

# ── Hoặc dùng curl ──────────────────────────
# curl -X POST https://xxxx.ngrok-free.app/predict \\
#      -H "Content-Type: application/json" \\
#      -d '{"text": "Patient has symptoms of pneumonia.", "num_beams": 4}'
'''

print(EXAMPLE_LOCAL_USAGE)


# ============================================================
# Chạy đoạn này trên MÁY LOCAL (không phải Kaggle)
# ============================================================
import requests

PUBLIC_URL = "https://xxxx-xxxx.ngrok-free.app"  # ← Thay bằng URL thực

response = requests.post(
    f"{PUBLIC_URL}/predict",
    json={
        "text": "Patient has symptoms of pneumonia.",
        "max_new_tokens": 256,
        "num_beams": 4,
        "length_penalty": 1.0,
    },
    headers={"Content-Type": "application/json"},
    timeout=60,
)

data = response.json()
print("Kết quả:", data["output"])
print("Latency:", data["latency_ms"], "ms")

# ── Hoặc dùng curl ──────────────────────────
# curl -X POST https://xxxx.ngrok-free.app/predict \
#      -H "Content-Type: application/json" \
#      -d '{"text": "Patient has symptoms of pneumonia.", "num_beams": 4}'

